# Automated Warehouse Robot Controller

This notebook outlines the design and implementation of a multi-agent control system for coordinating a fleet of warehouse robots. The system coordinates multiple robots to retrieve shelves and deliver them to packing stations without collisions or deadlocks. Various **Multi-Agent Pathfinding (MAPF)** techniques, including **Independent A***, **Cooperative A***, **Hill Climbing Optimization**, and **Conflict-Based Search (CBS)**, are employed to find collision-free paths that minimize both makespan and flowtime. The project includes comprehensive performance evaluation, deadlock detection, and visual simulation with heatmap analysis of warehouse congestion patterns.

## Data Collection & Research
We will be working on the Standard dataset by the time we collect a local one 

There are multiple choices for the standard dataset:
- **Small Grid**: 63*161
- **Medium Grid**: 84*170
- **Large Grid**: 123*321
- **Very Larg Grid**: 165*340

### Grid Envirenment Class
This class represents the warehouse as a grid-based environment that manages boundaries, obstacles, and valid robot movements.



In [ ]:
class GridEnvironment:
    """ Represents the grid map"""

    def __init__(self, filename):
        """ Load grid from file """
        self.grid = self.load_from_file(filename)

        """ Initialize grid with dimensions """
        self.height = len(self.grid)
        self.width = len(self.grid[0]) if self.height > 0 else 0

    def is_valid_position(self, x, y):
        """ Check if position is within grid bounds """
        return 0 <= x < self.width and 0 <= y < self.height

    def is_walkable(self, x, y):
        """ Check if position is walkable (not obstacle)"""
        return self.is_valid_position(x, y) and self.grid[y][x]

    def get_neighbors(self, x, y):
        """Get adjacent cells (4-directional)"""
        directions = [(0,1), (1,0), (0,-1), (-1,0)]
        neighbors = []

        for dx, dy in directions:
            nx, ny = x + dx, y + dy
            if self.is_walkable(nx, ny):
                neighbors.append((nx, ny))

        return neighbors

    def load_from_file(self, filename):
        """ Load grid from file """
        grid = []
        with open(filename, 'r') as f:
            for line in f:
                row = []
                for char in line.strip():
                    if char == '.':
                        row.append(True)
                    elif char == 'T':
                        row.append(False)
                grid.append(row)
        return grid

    def save_to_file(self, filename):
        """ Save grid to file """
        with open(filename, 'w') as f:
            for row in self.grid:
                line = ''.join(['.' if cell else 'T' for cell in row])
                f.write(line + '\n')

    def visualize(self):
        import matplotlib.pyplot as plt
        from matplotlib.colors import ListedColormap
        import numpy as np
        """Visualize the warehouse grid"""
        grid_array = np.array(self.grid, dtype=int)
        
        fig, ax = plt.subplots(figsize=(12, 12))
        
        cmap = ListedColormap(['black', 'white'])
        ax.imshow(grid_array, cmap=cmap, origin='upper', interpolation='nearest')
        
        ax.set_xticks(np.arange(-0.5, self.width, 1), minor=True)
        ax.set_yticks(np.arange(-0.5, self.height, 1), minor=True)
        
        ax.grid(which='minor', color='gray', linestyle='-', linewidth=0.3, alpha=0.5)
        
        ax.set_xticks(np.arange(0, self.width, 20))
        ax.set_yticks(np.arange(0, self.height, 20))
        

        ax.set_xlabel("X")
        ax.set_ylabel("Y")
        ax.set_title(f"Warehouse Grid ({self.width}x{self.height})")
        
        plt.tight_layout()
        plt.show()

small_grid = GridEnvironment('./maps/very-small.txt')
small_grid.visualize()

# medium_grid = GridEnvironment('./maps/medium.txt')
# medium_grid.visualize()

# large_grid = GridEnvironment('./maps/large.txt')
# large_grid.visualize()

# very_large_grid = GridEnvironment('./maps/very-large.txt')
# very_large_grid.visualize()



## Problem Definition

Given N robots at start positions and N goal positions (shelves to retrieve), find a set of
conflict-free paths such that all robots reach their goals in minimal time.

## Problem Formulation
### 1. State Representation

Each state is represented by a dictionary with the following structure:

```python
state = {
    
    # Robot
    'robots': [
        {
            'id': robot_id,                             # Unique identifier (0, 1, 2, ...)
            'start_position': (x0, y0),                 # Start postion (fixed)
            'goal_position': (gx, gy),                  # Goal position (fixed)
            'path': [(x0,y0), (x1,y1), ..., (gx, gy)],  # Full planned path (computed)
            'current_position': path[-1],               # Current position in the grid
            'at_goal': bool,                            # True if path[-1] == goal
        },
        # ... more robots
    ],
    
    # Global state
    'positions': {robot_id: (x, y)},          # Quick lookup: robot_id → position
    'collisions': count,                      # Cumulative collision count
    'deadlock': bool,                # True if circular wait detected
    'grid': GridEnvironment                   # Reference to warehouse layout
}
```

### 2. Goal Test

**Primary Goal**:
- No collisions occurred
- No deadlock detected
- All robots at their goal positions


**Success Criteria: (Detail)**
1. All robots reach goals 
2. No collisions/conflicts 
3. No deadlocks 
4. Minimum makespan  (secondary objective)
5. Minimum flowtime  (secondary objective)

### 3. Actions
- All robots move simultaneously, some could wait

- SubActions per robot: 
    - MoveUp: move up if walkable (path = [..., (x, y), (x - 1, y), ...]) 


    - MoveDown: move down if walkable (path = [..., (x, y), (x + 1, y), ...]) 


    - MoveLeft: move left if walkable (path = [..., (x, y), (x, y - 1), ...]) 


    - MoveRight: move right if walkable (path = [..., (x, y), (x, y + 1), ...]) 


    - Wait: stay in current position (path = [..., (x, y), (x, y), ...]) 



### 4. Transition Model
- The new state after applying the action:
    - The new position for each robot is added to its path, if it is a wait just add the last position to the path i.e. [..., (x, y), (x, y), ...]

    - Each robot may reach its goal_position, hence the attribute 'at_goal' may be updated 

    - The positions of the robots is changed

    - The number of collisions in the state is updated

    - The deadlock state is update (a deadlock may occur)

### 5. Path Cost

- Each move costs: 1 time unit
- Each wait costs: 1 time unit


### Initial State


**Properties of Initial State:**
- All robots at distinct start positions
- All start positions are walkable
- No paths planned yet
- No conflicts exist initially
- Time counter starts at 0
- Cost g(n) = 0 (no moves yet)


**Initial State Example:**
```python
initial_state = {
    'time_step': 0,
    'robots': [
        {'id': 1, 'start_position': (1, 1), 'goal_position': (8, 8), 'path': [(1, 1)], 'current_position': (1, 1), 'at_goal': False},
        {'id': 2, 'start_position': (8, 1), 'goal_position': (1, 8), 'path': [(8, 1)], 'current_position': (8, 1), 'at_goal': False},
        {'id': 3, 'start_position': (1, 8), 'goal_position': (8, 1), 'path': [(1, 8)], 'current_position': (1, 8), 'at_goal': False},
    ],
    'positions': {1: (1, 1), 2: (8, 1), 3: (1, 8)}
    'collisions': 0,
    'deadlock': False,
    'grid': GridEnvironment(filename)
}
```

- time_step is 0
- Robot 1: starts at (1,1), goal (8,8)
- Robot 2: starts at (8,1), goal (1,8)
- Robot 3: starts at (1,8), goal (8,1)
- Robots' positions are their starting positions
- No collisions 
- No deadlock
- Grid loaded from the file


### Robot Class

This class represents a single robot in the grid, managing its position, goal, and movement within the environment.

In [ ]:
class Robot :
    """ Represents a single robot """
    def __init__(self, grid, start_pos, goal_pos,color):
        """
        Initialize robot with GRID REFERENCE.
        
        Args:
            grid (GridEnvironment): The grid this robot moves in
            start_pos (tuple): Starting (x, y) position
            color (str): Robot color
        """
        # STORE GRID REFERENCE
        self.grid = grid
        pass

    def set_goal(self, goal_pos):
        """Set goal position for robot """
        pass

    def get_current_position(self):
        """Get current position """
        pass

    def get_goal_position(self):
        """Get goal position """
        pass

    def move_to(self ,position):
        """ Move robot to new position """
        pass

    def is_at_goal(self):
        """ Check if robot reached goal """
        pass

    def get_neighbors_from_grid(self):
        """Use grid method to get valid neighbor positions"""
        pass

    def get_path(self):
        """Get stored path """
        pass

    def set_path(self, path) :
        """ Store path for robot """
        pass


### Problem Class
Now, let's define the main Problem class that will encapsulate our MAPF problem

In [4]:
class AutomatedWarehouseRobotControllerProblem:
    pass

### Node Class
Now, let's define the Node class which will represent states in the search space

In [5]:
class Node:
    pass